# Task 3 — Audio Age, Gender, Senior Citizen & Emotion Detection

Cleaned from the original `emotion_predictions.ipynb`.

This notebook contains the complete workflow:

1. Common Voice age/gender dataset preparation
2. MFCC feature extraction
3. Age-group + gender model trained from scratch
4. Senior-citizen + gender model trained from scratch
5. RAVDESS emotion dataset preparation
6. Emotion CNN trained from scratch
7. Test evaluation
8. Save/load deployment models
9. Interactive Tkinter GUI

**No pretrained model is used.**

In [1]:
from pathlib import Path
import os
import glob
import numpy as np
import pandas as pd

COMMON_VOICE_ROOT = Path("/Users/nenavathdigambar/Downloads/common voice")
EMOTION_ROOT = Path("/Users/nenavathdigambar/Downloads/emotions")
MODEL_DIR = Path("/Users/nenavathdigambar/Downloads/audio_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Common Voice exists:", COMMON_VOICE_ROOT.exists())
print("Emotion dataset exists:", EMOTION_ROOT.exists())

Common Voice exists: True
Emotion dataset exists: True


In [2]:
# ============================================================
# COMMON VOICE — LOAD METADATA
# ============================================================

csv_files = [
    COMMON_VOICE_ROOT / "cv-valid-train.csv",
    COMMON_VOICE_ROOT / "cv-valid-test.csv",
    COMMON_VOICE_ROOT / "cv-valid-dev.csv",
    COMMON_VOICE_ROOT / "cv-other-train.csv",
    COMMON_VOICE_ROOT / "cv-other-test.csv",
    COMMON_VOICE_ROOT / "cv-other-dev.csv",
]

dfs = []

for csv_file in csv_files:
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        df["source_csv"] = csv_file.name
        dfs.append(df)
        print(csv_file.name, "→", df.shape)

cv_df = pd.concat(dfs, ignore_index=True)

print("\nCombined dataset:", cv_df.shape)
print("\nAge distribution:")
print(cv_df["age"].value_counts(dropna=False))

print("\nGender distribution:")
print(cv_df["gender"].value_counts(dropna=False))

cv-valid-train.csv → (195776, 9)
cv-valid-test.csv → (3995, 9)
cv-valid-dev.csv → (4076, 9)
cv-other-train.csv → (145135, 9)
cv-other-test.csv → (2961, 9)
cv-other-dev.csv → (3022, 9)

Combined dataset: (354965, 9)

Age distribution:
age
NaN          211795
twenties      45948
thirties      36810
fourties      21291
fifties       18466
teens          9278
sixties        8239
seventies      2706
eighties        432
Name: count, dtype: int64

Gender distribution:
gender
NaN       211552
male      108734
female     33469
other       1210
Name: count, dtype: int64


In [3]:
# ============================================================
# COMMON VOICE — CLEAN AGE + GENDER DATA
# ============================================================

age_mapping = {
    "teens": 18,
    "twenties": 24,
    "thirties": 34,
    "fourties": 44,
    "fifties": 54,
    "sixties": 64,
    "seventies": 74,
    "eighties": 84,
}

age_classes = {
    "teens": 0,
    "twenties": 1,
    "thirties": 2,
    "fourties": 3,
    "fifties": 4,
    "sixties": 5,
    "seventies": 6,
    "eighties": 7,
}

gender_map = {
    "female": 0,
    "male": 1
}

age_gender_df = cv_df.dropna(
    subset=["age", "gender"]
).copy()

age_gender_df = age_gender_df[
    age_gender_df["gender"].isin(["male", "female"])
].copy()

age_gender_df["age_numeric"] = age_gender_df["age"].map(age_mapping)
age_gender_df["age_class"] = age_gender_df["age"].map(age_classes)

age_gender_df = age_gender_df.dropna(
    subset=["age_numeric", "age_class"]
).copy()

age_gender_df["age_numeric"] = age_gender_df["age_numeric"].astype(int)
age_gender_df["age_class"] = age_gender_df["age_class"].astype(int)
age_gender_df["gender_label"] = (
    age_gender_df["gender"].map(gender_map).astype(int)
)

# Same senior definition used by the original training section:
# representative age >= 60.
age_gender_df["senior_citizen"] = (
    age_gender_df["age_numeric"] >= 60
).astype(int)

print("Clean dataset:", age_gender_df.shape)

print("\nGender:")
print(age_gender_df["gender"].value_counts())

print("\nAge:")
print(age_gender_df["age"].value_counts())

print("\nSenior:")
print(age_gender_df["senior_citizen"].value_counts())

Clean dataset: (141448, 13)

Gender:
gender
male      108224
female     33224
Name: count, dtype: int64

Age:
age
twenties     45152
thirties     36671
fourties     20809
fifties      18412
teens         9064
sixties       8202
seventies     2706
eighties       432
Name: count, dtype: int64

Senior:
senior_citizen
0    130108
1     11340
Name: count, dtype: int64


In [4]:
# ============================================================
# COMMON VOICE — MATCH AUDIO + REMOVE DUPLICATES
# ============================================================

audio_lookup = {}

for audio_file in COMMON_VOICE_ROOT.rglob("*.mp3"):
    audio_lookup[audio_file.name] = str(audio_file)

matched_df = age_gender_df.copy()

matched_df["audio_path"] = matched_df["filename"].apply(
    lambda x: audio_lookup.get(Path(x).name)
)

matched_df = matched_df.dropna(
    subset=["audio_path"]
).copy()

model_df = matched_df.drop_duplicates(
    subset=["audio_path"]
).copy()

train_df = model_df[
    model_df["source_csv"].str.contains("train")
].copy()

val_df = model_df[
    model_df["source_csv"].str.contains("dev")
].copy()

test_df = model_df[
    model_df["source_csv"].str.contains("test")
].copy()

print("Audio files found:", len(audio_lookup))
print("Matched:", len(matched_df))
print("Unique audio:", len(model_df))

print("\nTRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))

Audio files found: 195776
Matched: 141448
Unique audio: 113425

TRAIN: 111599
VALIDATION: 688
TEST: 1138


In [5]:
# Save clean Common Voice splits

train_df.to_csv(
    COMMON_VOICE_ROOT / "train_age_gender.csv",
    index=False
)

val_df.to_csv(
    COMMON_VOICE_ROOT / "val_age_gender.csv",
    index=False
)

test_df.to_csv(
    COMMON_VOICE_ROOT / "test_age_gender.csv",
    index=False
)

print("Saved Common Voice split CSV files.")

Saved Common Voice split CSV files.


In [6]:
# ============================================================
# IMPORTS + GPU CHECK
# ============================================================

import librosa
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve
)

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Librosa:", librosa.__version__)

gpus = tf.config.list_physical_devices("GPU")

print("\nAvailable devices:")
print(tf.config.list_physical_devices())

if gpus:
    print("\nGPU detected — training can use the GPU.")
else:
    print("\nNo GPU detected — training will use CPU.")

TensorFlow: 2.18.0
NumPy: 2.0.2
Librosa: 0.11.0

Available devices:
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

GPU detected — training can use the GPU.


In [7]:
# ============================================================
# COMMON VOICE — MFCC PREPROCESSING
# ============================================================

SAMPLE_RATE = 16000
N_MFCC = 40
MAX_FRAMES = 500

def extract_mfcc(file_path):
    try:
        audio, sr = librosa.load(
            file_path,
            sr=SAMPLE_RATE,
            mono=True
        )

        audio = librosa.util.normalize(audio)

        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sr,
            n_mfcc=N_MFCC,
            n_fft=512,
            hop_length=160
        )

        if mfcc.shape[1] < MAX_FRAMES:
            mfcc = np.pad(
                mfcc,
                ((0, 0), (0, MAX_FRAMES - mfcc.shape[1])),
                mode="constant"
            )
        else:
            mfcc = mfcc[:, :MAX_FRAMES]

        return mfcc.T.astype(np.float32)

    except Exception as e:
        print("Error:", file_path, e)
        return None

In [18]:
# ============================================================
# COMMON VOICE — TRAINING FEATURES
# ============================================================

# Keep the original notebook's 30,000-sample training subset
# to make training practical.

train_subset, _ = train_test_split(
    train_df,
    train_size=min(30000, len(train_df)),
    stratify=train_df["age_class"],
    random_state=42
)

train_subset = train_subset.reset_index(drop=True)

print("Training samples:", len(train_subset))
print("\nAge distribution:")
print(
    train_subset["age"]
    .value_counts()
    .reindex(age_classes.keys())
)

Training samples: 30000

Age distribution:
age
teens        1999
twenties     9480
thirties     7707
fourties     4440
fifties      3901
sixties      1784
seventies     595
eighties       94
Name: count, dtype: int64


In [19]:
# Extract training MFCCs once.
# These features are reused by both Common Voice models.
#
# NOTE: y_age is now the continuous age_numeric (bucket midpoint: 18, 24, 34, ...)
# instead of the age_class index, so the age head can regress a number instead
# of just picking one of 8 buckets.

def prepare_common_voice_features(df):

    X = []
    y_gender = []
    y_age = []
    y_senior = []

    for i, row in enumerate(df.itertuples()):

        features = extract_mfcc(row.audio_path)

        if features is not None:
            X.append(features)
            y_gender.append(row.gender_label)
            y_age.append(row.age_numeric)
            y_senior.append(row.senior_citizen)

        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1}/{len(df)}")

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y_gender, dtype=np.float32),
        np.asarray(y_age, dtype=np.float32),
        np.asarray(y_senior, dtype=np.float32)
    )


X_train, y_gender_train, y_age_train, y_senior_train = (
    prepare_common_voice_features(train_subset)
)

# Regression works far better on a normalized target (mean 0, std 1) than on
# raw ages in the teens-to-eighties range, especially since it shares a loss
# with the gender head. Save these stats - inference needs them to convert
# the model's normalized output back into a real age.
AGE_MEAN = float(y_age_train.mean())
AGE_STD = float(y_age_train.std())
y_age_train_norm = (y_age_train - AGE_MEAN) / AGE_STD

print("\nShapes:")
print("X_train:", X_train.shape)
print("Gender:", y_gender_train.shape)
print("Age:", y_age_train.shape)
print("Senior:", y_senior_train.shape)
print(f"\nAge normalization -> mean={AGE_MEAN:.2f}, std={AGE_STD:.2f}")


Processed 500/30000
Processed 1000/30000
Processed 1500/30000
Processed 2000/30000
Processed 2500/30000
Processed 3000/30000
Processed 3500/30000
Processed 4000/30000
Processed 4500/30000
Processed 5000/30000
Processed 5500/30000
Processed 6000/30000
Processed 6500/30000
Processed 7000/30000
Processed 7500/30000
Processed 8000/30000
Processed 8500/30000
Processed 9000/30000
Processed 9500/30000
Processed 10000/30000
Processed 10500/30000
Processed 11000/30000
Processed 11500/30000
Processed 12000/30000
Processed 12500/30000
Processed 13000/30000
Processed 13500/30000
Processed 14000/30000
Processed 14500/30000
Processed 15000/30000
Processed 15500/30000
Processed 16000/30000
Processed 16500/30000
Processed 17000/30000
Processed 17500/30000
Processed 18000/30000
Processed 18500/30000
Processed 19000/30000
Processed 19500/30000
Processed 20000/30000
Processed 20500/30000
Processed 21000/30000
Processed 21500/30000
Processed 22000/30000
Processed 22500/30000
Processed 23000/30000
Processe

In [20]:
# Extract the held-out Common Voice test set.

X_test_cv, y_gender_test, y_age_test, y_senior_test = (
    prepare_common_voice_features(test_df)
)

# Use the TRAINING set's mean/std (never recompute from test data - that
# would leak test-set statistics into the normalization).
y_age_test_norm = (y_age_test - AGE_MEAN) / AGE_STD

print("\nTest shape:", X_test_cv.shape)


Processed 500/1138
Processed 1000/1138

Test shape: (1138, 500, 40)


In [13]:
# ============================================================
# MODEL 1 — AGE (REGRESSION) + GENDER
# ============================================================

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Conv1D,
    BatchNormalization,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)

def build_age_gender_model():

    inp = Input(
        shape=(500, 40),
        name="mfcc_input"
    )

    x = Conv1D(
        64, 5,
        activation="relu",
        padding="same"
    )(inp)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = Conv1D(
        128, 5,
        activation="relu",
        padding="same"
    )(x)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = Conv1D(
        256, 3,
        activation="relu",
        padding="same"
    )(x)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = LSTM(128)(x)

    x = Dense(
        128,
        activation="relu"
    )(x)

    x = Dropout(0.4)(x)

    gender = Dense(
        1,
        activation="sigmoid",
        name="gender"
    )(x)

    # Regression head: one linear output, predicting the NORMALIZED age.
    # (raw age = output * AGE_STD + AGE_MEAN, done in predict_audio())
    age = Dense(
        1,
        activation="linear",
        name="age"
    )(x)

    model = Model(
        inp,
        [gender, age]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss={
            "gender": "binary_crossentropy",
            # Huber is MSE near zero, MAE further out - robust to the
            # occasional mislabeled/outlier age without being as harsh
            # as pure MSE on every small deviation.
            "age": keras.losses.Huber()
        },
        metrics={
            "gender": ["accuracy"],
            "age": ["mae"]
        }
    )

    return model


age_gender_model = build_age_gender_model()

age_gender_model.summary()


2026-08-15 20:54:54.301355: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-08-15 20:54:54.304689: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-08-15 20:54:54.304697: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1786807494.305366  315892 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1786807494.306665  315892 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mfcc_input          │ (None, 500, 40)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 500, 64)   │     12,864 │ mfcc_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 500, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 250, 128)  │     41,088 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 128)  │        512 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 125, 128)  │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 125, 256)  │     98,560 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 125, 256)  │      1,024 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 62, 256)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 128)       │    197,120 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (Dense)      │ (None, 1)         │        129 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ age (Dense)         │ (None, 1)         │        129 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 368,194 (1.40 MB)

 Trainable params: 367,298 (1.40 MB)

 Non-trainable params: 896 (3.50 KB)

In [14]:
# Train age + gender model

training_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

age_gender_history = age_gender_model.fit(
    X_train,
    {
        "gender": y_gender_train,
        "age": y_age_train_norm
    },
    validation_split=0.15,
    epochs=25,
    batch_size=64,
    shuffle=True,
    callbacks=training_callbacks
)


Epoch 1/25


2026-08-15 20:55:37.221387: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1483/1483 ━━━━━━━━━━━━━━━━━━━━ 72s 45ms/step - age_loss: 0.4306 - age_mae: 0.7996 - gender_accuracy: 0.7566 - gender_loss: 0.5977 - loss: 1.0284 - val_age_loss: 0.4273 - val_age_mae: 0.8005 - val_gender_accuracy: 0.7834 - val_gender_loss: 0.5429 - val_loss: 0.9701 - learning_rate: 0.0010
Epoch 2/25
1483/1483 ━━━━━━━━━━━━━━━━━━━━ 59s 40ms/step - age_loss: 0.4303 - age_mae: 0.7946 - gender_accuracy: 0.7570 - gender_loss: 0.5552 - loss: 0.9855 - val_age_loss: 0.4281 - val_age_mae: 0.8034 - val_gender_accuracy: 0.7838 - val_gender_loss: 0.5340 - val_loss: 0.9619 - learning_rate: 0.0010
Epoch 3/25
1483/1483 ━━━━━━━━━━━━━━━━━━━━ 60s 40ms/step - age_loss: 0.4302 - age_mae: 0.7952 - gender_accuracy: 0.7570 - gender_loss: 0.5543 - loss: 0.9847 - val_age_loss: 0.4273 - val_age_mae: 0.8006 - val_gender_accuracy: 0.7838 - val_gender_loss: 0.5335 - val_loss: 0.9608 - learning_rate: 0.0010
Epoch 4/25
1483/1483 ━━━━━━━━━━━━━━━━━━━━ 62s 42ms/step - age_loss: 0.4303 - age_mae: 0.7947 - gender_accuracy:

In [15]:
# Evaluate age + gender on the held-out Common Voice test set.

pred_gender, pred_age_norm = age_gender_model.predict(
    X_test_cv,
    batch_size=64,
    verbose=1
)

pred_gender_class = (
    pred_gender.ravel() >= 0.5
).astype(int)

# Undo the normalization to get back a real age in years.
pred_age = pred_age_norm.ravel() * AGE_STD + AGE_MEAN

print("GENDER RESULTS")
print("=" * 50)

print(
    classification_report(
        y_gender_test,
        pred_gender_class,
        target_names=["Female", "Male"],
        zero_division=0
    )
)

print("\nAGE REGRESSION RESULTS")
print("=" * 50)

age_errors = pred_age - y_age_test
mae = np.mean(np.abs(age_errors))
rmse = np.sqrt(np.mean(age_errors ** 2))
within_5 = np.mean(np.abs(age_errors) <= 5) * 100
within_10 = np.mean(np.abs(age_errors) <= 10) * 100

print(f"MAE:            {mae:.2f} years")
print(f"RMSE:           {rmse:.2f} years")
print(f"Within 5 years:  {within_5:.1f}%")
print(f"Within 10 years: {within_10:.1f}%")

print("\nSample predictions (true vs predicted):")
sample_idx = np.random.choice(len(y_age_test), size=min(10, len(y_age_test)), replace=False)
for i in sample_idx:
    print(f"  true={y_age_test[i]:5.0f}   pred={pred_age[i]:5.1f}")

print(
    "\nNote: Common Voice only labels age in ~10-year buckets, so this "
    "model was never shown a true single-year age during training - MAE "
    "and the sample predictions above are the honest ceiling on precision "
    "achievable with this dataset, not a bug to chase further."
)


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
GENDER RESULTS
              precision    recall  f1-score   support

      Female       0.00      0.00      0.00       263
        Male       0.77      1.00      0.87       875

    accuracy                           0.77      1138
   macro avg       0.38      0.50      0.43      1138
weighted avg       0.59      0.77      0.67      1138


AGE REGRESSION RESULTS
MAE:            11.37 years
RMSE:           14.47 years
Within 5 years:  25.6%
Within 10 years: 44.3%

Sample predictions (true vs predicted):
  true=   24   pred= 37.7
  true=   44   pred= 25.5
  true=   34   pred= 34.4
  true=   64   pred= 37.5
  true=   24   pred= 34.7
  true=   24   pred= 45.7
  true=   24   pred= 34.0
  true=   34   pred= 35.3
  true=   34   pred= 34.5
  true=   24   pred= 34.6

Note: Common Voice only labels age in ~10-year buckets, so this model was never shown a true single-year age during training - MAE and the sample predictions above are the honest ceiling on 

In [16]:
# ============================================================
# MODEL 2 — SENIOR CITIZEN + GENDER
# ============================================================

def build_senior_gender_model():

    inp = Input(
        shape=(500, 40),
        name="mfcc_input"
    )

    x = Conv1D(
        64, 5,
        activation="relu",
        padding="same"
    )(inp)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = Conv1D(
        128, 5,
        activation="relu",
        padding="same"
    )(x)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = Conv1D(
        256, 3,
        activation="relu",
        padding="same"
    )(x)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = LSTM(128)(x)

    x = Dense(
        128,
        activation="relu"
    )(x)

    x = Dropout(0.4)(x)

    gender = Dense(
        1,
        activation="sigmoid",
        name="gender"
    )(x)

    senior = Dense(
        1,
        activation="sigmoid",
        name="senior"
    )(x)

    return Model(
        inp,
        [gender, senior]
    )


senior_gender_model = build_senior_gender_model()

n_non_senior = np.sum(
    y_senior_train == 0
)

n_senior = np.sum(
    y_senior_train == 1
)

senior_weight = (
    n_non_senior /
    max(n_senior, 1)
)

print(
    "Senior positive-class weight:",
    round(float(senior_weight), 3)
)


def weighted_binary_crossentropy(pos_weight):

    def loss(y_true, y_pred):

        y_true = tf.cast(
            y_true,
            tf.float32
        )

        y_pred = tf.clip_by_value(
            y_pred,
            keras.backend.epsilon(),
            1 - keras.backend.epsilon()
        )

        value = -(
            pos_weight *
            y_true *
            tf.math.log(y_pred)
            +
            (1 - y_true) *
            tf.math.log(1 - y_pred)
        )

        return tf.reduce_mean(value)

    return loss


senior_gender_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss={
        "gender": "binary_crossentropy",
        "senior": weighted_binary_crossentropy(
            senior_weight
        )
    },
    metrics={
        "gender": ["accuracy"],
        "senior": ["accuracy"]
    }
)

senior_gender_model.summary()

Senior positive-class weight: 11.133


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mfcc_input          │ (None, 500, 40)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 500, 64)   │     12,864 │ mfcc_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 500, 64)   │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 250, 64)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 250, 128)  │     41,088 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 250, 128)  │        512 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_4     │ (None, 125, 128)  │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 125, 256)  │     98,560 │ max_pooling1d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 125, 256)  │      1,024 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_5     │ (None, 62, 256)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 128)       │    197,120 │ max_pooling1d_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (Dense)      │ (None, 1)         │        129 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ senior (Dense)      │ (None, 1)         │        129 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 368,194 (1.40 MB)

 Trainable params: 367,298 (1.40 MB)

 Non-trainable params: 896 (3.50 KB)

In [17]:
# Train senior + gender model

senior_history = senior_gender_model.fit(
    X_train,
    {
        "gender": y_gender_train,
        "senior": y_senior_train
    },
    validation_split=0.15,
    epochs=20,
    batch_size=64,
    shuffle=True,
    callbacks=training_callbacks
)

Epoch 1/20
1483/1483 ━━━━━━━━━━━━━━━━━━━━ 69s 44ms/step - gender_accuracy: 0.7567 - gender_loss: 0.5974 - loss: 1.8840 - senior_accuracy: 0.2494 - senior_loss: 1.2865 - val_gender_accuracy: 0.7841 - val_gender_loss: 0.5318 - val_loss: 1.8328 - val_senior_accuracy: 0.0813 - val_senior_loss: 1.3015 - learning_rate: 0.0010
Epoch 2/20
1483/1483 ━━━━━━━━━━━━━━━━━━━━ 62s 42ms/step - gender_accuracy: 0.7570 - gender_loss: 0.5551 - loss: 1.8417 - senior_accuracy: 0.0933 - senior_loss: 1.2862 - val_gender_accuracy: 0.7842 - val_gender_loss: 0.5277 - val_loss: 1.8296 - val_senior_accuracy: 0.0810 - val_senior_loss: 1.3024 - learning_rate: 0.0010
Epoch 3/20
 302/1483 ━━━━━━━━━━━━━━━━━━━━ 49s 42ms/step - gender_accuracy: 0.7556 - gender_loss: 0.5561 - loss: 1.8409 - senior_accuracy: 0.0842 - senior_loss: 1.2847

KeyboardInterrupt: 

In [ ]:
# Evaluate senior model and find the F1-based threshold.

pred_gender_s, pred_senior = senior_gender_model.predict(
    X_test_cv,
    batch_size=64,
    verbose=1
)

pred_gender_s_class = (
    pred_gender_s.ravel() >= 0.5
).astype(int)

print("GENDER")
print("=" * 50)

print(
    classification_report(
        y_gender_test,
        pred_gender_s_class,
        target_names=["Female", "Male"],
        zero_division=0
    )
)

precision, recall, thresholds = precision_recall_curve(
    y_senior_test,
    pred_senior.ravel()
)

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    /
    (
        precision[:-1] +
        recall[:-1] +
        1e-8
    )
)

best_idx = int(np.argmax(f1_scores))

SENIOR_THRESHOLD = float(
    thresholds[best_idx]
)

pred_senior_class = (
    pred_senior.ravel() >= SENIOR_THRESHOLD
).astype(int)

print("\nSENIOR CITIZEN")
print("=" * 50)

print(
    classification_report(
        y_senior_test,
        pred_senior_class,
        target_names=["Not Senior", "Senior"],
        zero_division=0
    )
)

print(
    "Best threshold:",
    round(SENIOR_THRESHOLD, 3)
)

print(
    "Best F1:",
    round(float(f1_scores[best_idx]), 3)
)

print("\nCONFUSION MATRIX")
print(
    confusion_matrix(
        y_senior_test,
        pred_senior_class
    )
)

In [ ]:
# ============================================================
# EMOTION DATASET — RAVDESS
# ============================================================

emotion_map = {
    1: "neutral",
    2: "calm",
    3: "happy",
    4: "sad",
    5: "angry",
    6: "fearful",
    7: "disgust",
    8: "surprised"
}

emotion_files = glob.glob(
    str(EMOTION_ROOT / "Actor_*" / "*.wav")
)

emotion_data = []

for path in emotion_files:

    parts = Path(path).stem.split("-")
    emotion_code = int(parts[2])

    emotion_data.append({
        "audio_path": path,
        "emotion_code": emotion_code,
        "emotion": emotion_map[emotion_code]
    })

emotion_df = pd.DataFrame(
    emotion_data
)

print("Total WAV files:", len(emotion_df))
print("\nEmotion distribution:")
print(
    emotion_df["emotion"].value_counts()
)

In [ ]:
# Split emotion data

train_emotion, temp_emotion = train_test_split(
    emotion_df,
    test_size=0.30,
    stratify=emotion_df["emotion_code"],
    random_state=42
)

val_emotion, test_emotion = train_test_split(
    temp_emotion,
    test_size=0.50,
    stratify=temp_emotion["emotion_code"],
    random_state=42
)

train_emotion = train_emotion.reset_index(drop=True)
val_emotion = val_emotion.reset_index(drop=True)
test_emotion = test_emotion.reset_index(drop=True)

print("Training:", len(train_emotion))
print("Validation:", len(val_emotion))
print("Testing:", len(test_emotion))

In [ ]:
# Emotion MFCC extraction

EMOTION_MAX_FRAMES = 300

def extract_emotion_mfcc(file_path):

    try:
        audio, sr = librosa.load(
            file_path,
            sr=16000,
            mono=True
        )

        audio = librosa.util.normalize(
            audio
        )

        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sr,
            n_mfcc=40,
            n_fft=512,
            hop_length=160
        )

        if mfcc.shape[1] < EMOTION_MAX_FRAMES:

            mfcc = np.pad(
                mfcc,
                (
                    (0, 0),
                    (
                        0,
                        EMOTION_MAX_FRAMES -
                        mfcc.shape[1]
                    )
                ),
                mode="constant"
            )

        else:
            mfcc = mfcc[
                :, :EMOTION_MAX_FRAMES
            ]

        return mfcc.T.astype(
            np.float32
        )

    except Exception as e:

        print(
            "Error:",
            file_path,
            e
        )

        return None


def prepare_emotion_data(df):

    X = []
    y = []

    for i, row in enumerate(
        df.itertuples()
    ):

        features = extract_emotion_mfcc(
            row.audio_path
        )

        if features is not None:

            X.append(features)

            # 1–8 -> 0–7
            y.append(
                row.emotion_code - 1
            )

        if (i + 1) % 100 == 0:
            print(
                f"Processed {i + 1}/{len(df)}"
            )

    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            y,
            dtype=np.int32
        )
    )


X_emotion_train, y_emotion_train = (
    prepare_emotion_data(
        train_emotion
    )
)

X_emotion_val, y_emotion_val = (
    prepare_emotion_data(
        val_emotion
    )
)

X_emotion_test, y_emotion_test = (
    prepare_emotion_data(
        test_emotion
    )
)

print("\nShapes:")
print(
    "Train:",
    X_emotion_train.shape,
    y_emotion_train.shape
)

print(
    "Validation:",
    X_emotion_val.shape,
    y_emotion_val.shape
)

print(
    "Test:",
    X_emotion_test.shape,
    y_emotion_test.shape
)

In [ ]:
# ============================================================
# MODEL 3 — EMOTION CNN FROM SCRATCH
# ============================================================

def build_emotion_model():

    inp = Input(
        shape=(300, 40),
        name="mfcc_input"
    )

    x = Conv1D(
        64, 5,
        padding="same",
        activation="relu"
    )(inp)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = Conv1D(
        128, 5,
        padding="same",
        activation="relu"
    )(x)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = Conv1D(
        256, 3,
        padding="same",
        activation="relu"
    )(x)

    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = keras.layers.GlobalAveragePooling1D()(x)

    x = Dense(
        128,
        activation="relu"
    )(x)

    x = Dropout(0.4)(x)

    out = Dense(
        8,
        activation="softmax",
        name="emotion"
    )(x)

    model = Model(
        inp,
        out
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


emotion_model = build_emotion_model()

emotion_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

emotion_history = emotion_model.fit(
    X_emotion_train,
    y_emotion_train,
    validation_data=(
        X_emotion_val,
        y_emotion_val
    ),
    epochs=40,
    batch_size=32,
    shuffle=True,
    callbacks=emotion_callbacks
)

In [ ]:
# Evaluate emotion model on the unseen test set

emotion_loss, emotion_accuracy = (
    emotion_model.evaluate(
        X_emotion_test,
        y_emotion_test,
        batch_size=32,
        verbose=1
    )
)

pred_emotions = emotion_model.predict(
    X_emotion_test,
    batch_size=32,
    verbose=0
)

pred_emotion_class = np.argmax(
    pred_emotions,
    axis=1
)

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised"
]

print(
    "Test accuracy:",
    round(float(emotion_accuracy), 4)
)

print(
    classification_report(
        y_emotion_test,
        pred_emotion_class,
        target_names=emotion_names,
        zero_division=0
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        y_emotion_test,
        pred_emotion_class
    )
)

In [ ]:
# ============================================================
# SAVE ALL MODELS
# ============================================================

age_gender_model.save(
    MODEL_DIR /
    "age_gender_audio_model.keras"
)

senior_gender_model.save(
    MODEL_DIR /
    "senior_gender_audio_model.keras"
)

emotion_model.save(
    MODEL_DIR /
    "emotion_audio_model.keras"
)

np.save(
    MODEL_DIR /
    "senior_threshold.npy",
    np.array(
        [SENIOR_THRESHOLD],
        dtype=np.float32
    )
)

# Needed at inference time to convert the age model's normalized output
# back into a real age in years.
np.save(
    MODEL_DIR /
    "age_norm_stats.npy",
    np.array(
        [AGE_MEAN, AGE_STD],
        dtype=np.float32
    )
)

print(
    "Models saved to:",
    MODEL_DIR
)


In [ ]:
# ============================================================
# LOAD MODELS FOR DEPLOYMENT
# ============================================================

age_gender_model = keras.models.load_model(
    MODEL_DIR /
    "age_gender_audio_model.keras",
    compile=False
)

senior_gender_model = keras.models.load_model(
    MODEL_DIR /
    "senior_gender_audio_model.keras",
    compile=False
)

emotion_model = keras.models.load_model(
    MODEL_DIR /
    "emotion_audio_model.keras",
    compile=False
)

SENIOR_THRESHOLD = float(
    np.load(
        MODEL_DIR /
        "senior_threshold.npy"
    )[0]
)

_age_stats = np.load(
    MODEL_DIR /
    "age_norm_stats.npy"
)
AGE_MEAN = float(_age_stats[0])
AGE_STD = float(_age_stats[1])

print("Deployment models loaded.")
print(
    "Senior threshold:",
    round(SENIOR_THRESHOLD, 3)
)
print(
    "Age normalization -> mean:", round(AGE_MEAN, 2),
    " std:", round(AGE_STD, 2)
)


In [ ]:
# ============================================================
# FINAL PREDICTION FUNCTION
# ============================================================

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised"
]

def pad_or_crop(mfcc, frames):

    if mfcc.shape[1] < frames:

        mfcc = np.pad(
            mfcc,
            (
                (0, 0),
                (0, frames - mfcc.shape[1])
            ),
            mode="constant"
        )

    else:
        mfcc = mfcc[
            :, :frames
        ]

    return mfcc


def predict_audio(audio_path):

    audio, sr = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    audio = librosa.util.normalize(
        audio
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40,
        n_fft=512,
        hop_length=160
    )

    # -------------------------------
    # AGE (regression) + GENDER
    # -------------------------------

    x500 = pad_or_crop(
        mfcc.copy(),
        500
    ).T.astype(np.float32)

    x500 = np.expand_dims(
        x500,
        axis=0
    )

    gender_pred, age_pred_norm = (
        age_gender_model.predict(
            x500,
            verbose=0
        )
    )

    gender_prob = float(
        gender_pred[0][0]
    )

    gender = (
        "Male"
        if gender_prob >= 0.5
        else "Female"
    )

    gender_confidence = (
        gender_prob
        if gender == "Male"
        else 1 - gender_prob
    )

    # Undo training-time normalization to get a real age in years.
    predicted_age = (
        float(age_pred_norm[0][0]) * AGE_STD + AGE_MEAN
    )
    predicted_age = max(0.0, predicted_age)

    # -------------------------------
    # SENIOR
    # -------------------------------

    _, senior_pred = (
        senior_gender_model.predict(
            x500,
            verbose=0
        )
    )

    senior_prob = float(
        senior_pred[0][0]
    )

    senior = (
        senior_prob >=
        SENIOR_THRESHOLD
    )

    # -------------------------------
    # EMOTION
    # -------------------------------

    x300 = pad_or_crop(
        mfcc.copy(),
        300
    ).T.astype(np.float32)

    x300 = np.expand_dims(
        x300,
        axis=0
    )

    emotion_probs = (
        emotion_model.predict(
            x300,
            verbose=0
        )[0]
    )

    emotion_index = int(
        np.argmax(
            emotion_probs
        )
    )

    top3_indices = np.argsort(
        emotion_probs
    )[::-1][:3]

    top3 = [
        (
            emotion_names[int(i)],
            float(emotion_probs[int(i)])
        )
        for i in top3_indices
    ]

    return {
        "file": os.path.basename(
            audio_path
        ),
        "gender": gender,
        "gender_confidence":
            float(gender_confidence),
        "predicted_age":
            round(predicted_age, 1),
        "senior_citizen":
            "YES" if senior else "NO",
        "senior_probability":
            senior_prob,
        "emotion":
            emotion_names[
                emotion_index
            ],
        "emotion_confidence":
            float(
                emotion_probs[
                    emotion_index
                ]
            ),
        "top3_emotions": top3
    }


In [ ]:
# ============================================================
# INTERACTIVE TASK 3 GUI
# ============================================================
#
# Pure presentation layer — all prediction logic lives in the
# `predict_audio()` cell above. This cell only calls it and displays:
#   Age, Gender, Senior Citizenship, Emotion
# (no raw confidences / probabilities / top-3 breakdown cluttering the UI)
#
# Interactive features:
#   - Waveform preview drawn right after upload
#   - Play / Stop with a live elapsed-time readout (Space bar shortcut)
#   - Animated progress bar while analyzing (background thread)
#   - Four big colour-coded result cards (Age, Gender, Senior, Emotion)
#   - History tab logging every analysis this session
#
# macOS playback uses the built-in `afplay` command.
# ============================================================

import os
import time
import datetime
import subprocess
import threading
import tkinter as tk
from tkinter import filedialog, messagebox, ttk

import numpy as np
import librosa
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

BG = "#1e1f26"
PANEL = "#262832"
CARD = "#2c2f3a"
ACCENT = "#4c8bf5"
GREEN = "#4fd06a"
AMBER = "#f5b942"
PURPLE = "#b98cf0"
TEXT = "#eaeaf0"
MUTED = "#9797a8"

CARD_SPECS = [
    ("predicted_age",  "🎂  Age",              ACCENT),
    ("gender",          "🚻  Gender",           GREEN),
    ("senior_citizen",  "👴  Senior Citizen",   AMBER),
    ("emotion",         "🎭  Emotion",          PURPLE),
]


class ResultCard(tk.Frame):
    """One big stat card: icon/title on top, large value below."""

    def __init__(self, parent, title, color):
        super().__init__(parent, bg=CARD, highlightbackground=color,
                          highlightthickness=2, bd=0)
        tk.Label(self, text=title, font=("Arial", 12, "bold"),
                 bg=CARD, fg=MUTED).pack(pady=(14, 2))
        self.value_var = tk.StringVar(value="—")
        tk.Label(self, textvariable=self.value_var, font=("Arial", 20, "bold"),
                 bg=CARD, fg=color, wraplength=200, justify="center").pack(pady=(0, 16))

    def set(self, text):
        self.value_var.set(text)

    def reset(self):
        self.value_var.set("—")


class AudioAnalysisApp:

    def __init__(self, root):
        self.root = root
        self.root.title("Task 3 — Audio Age, Gender & Emotion Analysis")
        self.root.geometry("960x720")
        self.root.minsize(880, 660)
        self.root.configure(bg=BG)

        self.audio_path = None
        self.player = None
        self.play_start_time = None
        self.play_duration = 0.0
        self._play_tick_job = None
        self.history = []

        self._build_style()
        self._build_layout()
        self.root.bind("<space>", self._on_space)

    # -------------------------------------------------- style / layout
    def _build_style(self):
        style = ttk.Style(self.root)
        try:
            style.theme_use("clam")
        except tk.TclError:
            pass
        style.configure("TNotebook", background=BG, borderwidth=0)
        style.configure("TNotebook.Tab", background=PANEL, foreground=TEXT,
                         font=("Arial", 11, "bold"), padding=(16, 8))
        style.map("TNotebook.Tab", background=[("selected", ACCENT)])
        style.configure("Treeview", background=CARD, fieldbackground=CARD,
                         foreground=TEXT, rowheight=26, font=("Arial", 10))
        style.configure("Treeview.Heading", background=PANEL, foreground=TEXT,
                         font=("Arial", 10, "bold"))
        style.configure("Horizontal.TProgressbar", background=ACCENT, troughcolor=CARD)

    def _build_layout(self):
        header = tk.Frame(self.root, bg=BG)
        header.pack(fill="x", pady=(18, 6))
        tk.Label(header, text="🎙 Audio Age, Gender & Emotion Analysis",
                 font=("Arial", 22, "bold"), bg=BG, fg=TEXT).pack()
        tk.Label(header, text="Upload a voice note — press Space to play/stop, then Analyze",
                 font=("Arial", 11), bg=BG, fg=MUTED).pack(pady=(2, 0))

        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill="both", expand=True, padx=16, pady=12)

        self.analyze_tab = tk.Frame(self.notebook, bg=BG)
        self.history_tab = tk.Frame(self.notebook, bg=BG)
        self.notebook.add(self.analyze_tab, text="  Analyze  ")
        self.notebook.add(self.history_tab, text="  History  ")

        self._build_analyze_tab()
        self._build_history_tab()

    # -------------------------------------------------- analyze tab
    def _build_analyze_tab(self):
        parent = self.analyze_tab

        controls = tk.Frame(parent, bg=BG)
        controls.pack(pady=(10, 4))

        tk.Button(controls, text="📁 Upload Audio", font=("Arial", 12, "bold"),
                   bg=ACCENT, fg="white", relief="flat", padx=14, pady=6,
                   command=self.upload_audio).grid(row=0, column=0, padx=5)

        self.play_btn = tk.Button(controls, text="▶ Play", font=("Arial", 12),
                                   bg=CARD, fg=TEXT, relief="flat", padx=14, pady=6,
                                   command=self.toggle_play, state="disabled")
        self.play_btn.grid(row=0, column=1, padx=5)

        self.analyze_btn = tk.Button(controls, text="🔍 Analyze", font=("Arial", 12, "bold"),
                                      bg=GREEN, fg="white", relief="flat", padx=14, pady=6,
                                      command=self.start_analysis, state="disabled")
        self.analyze_btn.grid(row=0, column=2, padx=5)

        tk.Button(controls, text="Clear", font=("Arial", 12), bg=CARD, fg=TEXT,
                   relief="flat", padx=14, pady=6,
                   command=self.clear_results).grid(row=0, column=3, padx=5)

        info_row = tk.Frame(parent, bg=BG)
        info_row.pack(pady=(10, 0))
        self.file_var = tk.StringVar(value="No audio selected")
        tk.Label(info_row, textvariable=self.file_var, font=("Arial", 11),
                 bg=BG, fg=TEXT).pack()
        self.elapsed_var = tk.StringVar(value="")
        tk.Label(info_row, textvariable=self.elapsed_var, font=("Arial", 10),
                 bg=BG, fg=MUTED).pack()

        # waveform preview
        wf_frame = tk.Frame(parent, bg=PANEL)
        wf_frame.pack(fill="x", padx=40, pady=(12, 6))
        self.fig = Figure(figsize=(7.5, 1.3), dpi=100, facecolor=PANEL)
        self.ax = self.fig.add_subplot(111)
        self._style_waveform_axes()
        self.canvas_wave = FigureCanvasTkAgg(self.fig, master=wf_frame)
        self.canvas_wave.get_tk_widget().pack(fill="x", padx=8, pady=8)

        # status + progress
        self.status_var = tk.StringVar(value="Ready")
        tk.Label(parent, textvariable=self.status_var, font=("Arial", 11, "bold"),
                 bg=BG, fg=ACCENT).pack(pady=(6, 2))
        self.progress = ttk.Progressbar(parent, mode="indeterminate", length=300)
        self.progress.pack(pady=(0, 14))

        # four result cards, only the required outputs
        cards_frame = tk.Frame(parent, bg=BG)
        cards_frame.pack(padx=40, pady=(0, 20))
        self.cards = {}
        for col, (key, title, color) in enumerate(CARD_SPECS):
            card = ResultCard(cards_frame, title, color)
            card.grid(row=0, column=col, padx=10, sticky="nsew")
            cards_frame.columnconfigure(col, weight=1)
            self.cards[key] = card

    def _style_waveform_axes(self):
        self.ax.clear()
        self.ax.set_facecolor(PANEL)
        self.ax.axis("off")

    # -------------------------------------------------- history tab
    def _build_history_tab(self):
        parent = self.history_tab
        tk.Label(parent, text="Session History", font=("Arial", 14, "bold"),
                 bg=BG, fg=TEXT).pack(pady=(14, 6))

        columns = ("time", "file", "age_group", "gender", "senior", "emotion")
        self.tree = ttk.Treeview(parent, columns=columns, show="headings", height=16)
        for col, label, w in [
            ("time", "Time", 80), ("file", "File", 200), ("age_group", "Age", 100),
            ("gender", "Gender", 90), ("senior", "Senior", 90), ("emotion", "Emotion", 110),
        ]:
            self.tree.heading(col, text=label)
            self.tree.column(col, width=w, anchor="w")
        self.tree.pack(fill="both", expand=True, padx=30, pady=14)

    def _add_history(self, result):
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        self.history.append(result)
        self.tree.insert("", "end", values=(
            ts, result["file"], f"{result['predicted_age']:.0f} yrs",
            result["gender"], result["senior_citizen"], result["emotion"].capitalize(),
        ))

    # -------------------------------------------------- upload
    def upload_audio(self):
        path = filedialog.askopenfilename(
            title="Select Audio",
            filetypes=[("Audio files", "*.wav *.mp3"), ("WAV files", "*.wav"),
                       ("MP3 files", "*.mp3")],
        )
        if not path:
            return

        self.stop_audio()
        self.audio_path = path
        self.file_var.set("Selected: " + os.path.basename(path))
        self.status_var.set("Audio selected — press Space or Play to listen")
        self.play_btn.config(state="normal")
        self.analyze_btn.config(state="normal")
        self.clear_prediction_labels()
        self._draw_waveform(path)

    def _draw_waveform(self, path):
        try:
            audio, sr = librosa.load(path, sr=8000, mono=True)
            self.play_duration = len(audio) / sr
            self._style_waveform_axes()
            t = np.linspace(0, self.play_duration, num=len(audio))
            self.ax.plot(t, audio, color=ACCENT, linewidth=0.6)
            self.ax.set_xlim(0, max(self.play_duration, 0.01))
            self.fig.tight_layout(pad=0.2)
            self.canvas_wave.draw()
        except Exception:
            pass  # waveform preview is best-effort; analysis still works if this fails

    # -------------------------------------------------- play / stop
    def toggle_play(self):
        if self.player is not None:
            self.stop_audio()
        else:
            self.play_audio()

    def _on_space(self, event):
        if self.audio_path:
            self.toggle_play()

    def play_audio(self):
        if not self.audio_path:
            return
        self.stop_audio()
        try:
            self.player = subprocess.Popen(["afplay", self.audio_path])
            self.play_btn.config(text="■ Stop")
            self.status_var.set("Playing audio...")
            self.play_start_time = time.time()
            self._tick_elapsed()
        except Exception as e:
            messagebox.showerror("Playback Error", f"Could not play this audio file.\n\n{e}")

    def _tick_elapsed(self):
        if self.player is None or self.play_start_time is None:
            return
        elapsed = time.time() - self.play_start_time
        total = self.play_duration or 0
        self.elapsed_var.set(f"{elapsed:0.1f}s / {total:0.1f}s")
        if self.player.poll() is not None or elapsed >= total + 0.3:
            self._on_playback_finished()
            return
        self._play_tick_job = self.root.after(150, self._tick_elapsed)

    def _on_playback_finished(self):
        self.player = None
        self.play_start_time = None
        self.play_btn.config(text="▶ Play")
        self.elapsed_var.set("")
        if self.audio_path:
            self.status_var.set("Audio selected")

    def stop_audio(self):
        if self._play_tick_job is not None:
            self.root.after_cancel(self._play_tick_job)
            self._play_tick_job = None
        if self.player is not None:
            try:
                self.player.terminate()
            except Exception:
                pass
        self.player = None
        self.play_start_time = None
        self.play_btn.config(text="▶ Play")
        self.elapsed_var.set("")
        if self.audio_path:
            self.status_var.set("Audio selected")

    # -------------------------------------------------- analyze
    def start_analysis(self):
        if not self.audio_path:
            return
        self.analyze_btn.config(state="disabled")
        self.status_var.set("Analyzing audio...")
        self.progress.start(12)
        threading.Thread(target=self.run_prediction, daemon=True).start()

    def run_prediction(self):
        try:
            result = predict_audio(self.audio_path)
            self.root.after(0, lambda: self.show_result(result))
        except Exception as e:
            self.root.after(0, lambda: messagebox.showerror("Prediction Error", str(e)))
            self.root.after(0, lambda: self.status_var.set("Prediction failed"))
        finally:
            self.root.after(0, lambda: self.progress.stop())
            self.root.after(0, lambda: self.analyze_btn.config(state="normal"))

    def show_result(self, result):
        self.cards["predicted_age"].set(f"{result['predicted_age']:.0f} yrs")
        self.cards["gender"].set(result["gender"])
        self.cards["senior_citizen"].set(result["senior_citizen"])
        self.cards["emotion"].set(result["emotion"].capitalize())

        self._add_history(result)
        self.status_var.set("Analysis completed")

    # -------------------------------------------------- clear
    def clear_prediction_labels(self):
        for card in self.cards.values():
            card.reset()

    def clear_results(self):
        self.stop_audio()
        self.audio_path = None
        self.file_var.set("No audio selected")
        self.status_var.set("Ready")
        self.elapsed_var.set("")
        self.play_btn.config(state="disabled")
        self.analyze_btn.config(state="disabled")
        self.clear_prediction_labels()
        self._style_waveform_axes()
        self.canvas_wave.draw()


# ============================================================
# RUN GUI
# ============================================================

root = tk.Tk()
app = AudioAnalysisApp(root)
root.protocol("WM_DELETE_WINDOW", lambda: (app.stop_audio(), root.destroy()))
root.mainloop()
